# CS-4063 â€” Natural Language Processing, Assignment 3
## Transformer-based Review Understanding with RAG Enhanced Explanation Generation

**Student ID:** i232545  
**Course:** CS-4063 Natural Language Processing  
**Due Date:** 29-04-2026

---

## Project Setup + Dataset Loading + Preprocessing Pipeline

**This commit covers:**
- Project directory setup
- Dataset loading from `.json.gz` files
- Category sampling (10kâ€“15k per category, 3 categories)
- Full preprocessing pipeline (cleaning â†’ tokenization â†’ vocabulary â†’ numericalization â†’ padding/truncation)
- Train / Validation / Test split (70 / 15 / 15)
- Saving preprocessed data and vocabulary to disk

---
## Section 0: Imports & Global Configuration

In [1]:
# â”€â”€ Standard library â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os
import re
import json
import gzip
import random
import pickle
import string
from collections import Counter
from pathlib import Path

# â”€â”€ Third-party â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# â”€â”€ Reproducibility â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports successful.")

All imports successful.


In [2]:
# â”€â”€ Project-wide constants â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# Paths (relative to notebook location)
DATASET_DIR   = Path("Dataset")
MODELS_DIR    = Path("models")
RESULTS_DIR   = Path("results")
PLOTS_DIR     = RESULTS_DIR / "learning_curves"

# Dataset construction
CATEGORIES = {
    "sports":      DATASET_DIR / "sports.json.gz",
    "beauty":      DATASET_DIR / "beauty.json.gz",
    "cellphones":  DATASET_DIR / "cellphones.json.gz",
}
SAMPLES_PER_CATEGORY = 12000   # target ~12k per category â†’ ~36k total

# Preprocessing
MAX_SEQ_LEN  = 128             # max tokens per review (truncate/pad to this)
MIN_FREQ     = 2               # minimum token frequency to enter vocabulary

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Special tokens
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]

# Sentiment label mapping
SENTIMENT_MAP = {
    1: 0,  # Negative
    2: 0,  # Negative
    3: 1,  # Neutral
    4: 2,  # Positive
    5: 2,  # Positive
}
SENTIMENT_LABELS = ["Negative", "Neutral", "Positive"]

# Create output directories
for d in [MODELS_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Dataset dir   : {DATASET_DIR.resolve()}")
print(f"  Models dir    : {MODELS_DIR.resolve()}")
print(f"  Results dir   : {RESULTS_DIR.resolve()}")
print(f"  Samples/cat   : {SAMPLES_PER_CATEGORY}")
print(f"  Max seq len   : {MAX_SEQ_LEN}")

Configuration loaded.
  Dataset dir   : C:\Users\DELL\Documents\NLP Assignment 3\Dataset
  Models dir    : C:\Users\DELL\Documents\NLP Assignment 3\models
  Results dir   : C:\Users\DELL\Documents\NLP Assignment 3\results
  Samples/cat   : 12000
  Max seq len   : 128


---
## Section 1: Dataset Loading

The Amazon Reviews dataset is stored as `.json.gz` files where each line is a separate JSON object. We load each category file, extract the `reviewText` and `overall` (star rating) fields, drop rows with missing values, and sample the required number of reviews.

In [3]:
def load_category(filepath: Path, n_samples: int, category_name: str) -> pd.DataFrame:
    """
    Load up to n_samples reviews from a .json.gz Amazon Reviews file.
    Only keeps rows with non-empty reviewText and a valid star rating (1-5).

    Args:
        filepath     : path to the .json.gz file
        n_samples    : maximum number of rows to sample
        category_name: label stored in the 'category' column

    Returns:
        DataFrame with columns [review_text, rating, sentiment, category]
    """
    records = []
    print(f"Loading '{category_name}' from {filepath} ...", flush=True)

    with gzip.open(filepath, "rt", encoding="utf-8", errors="ignore") as f:
        for line in tqdm(f, desc=f"  Reading {category_name}", unit=" lines"):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            text   = obj.get("reviewText", "") or ""
            rating = obj.get("overall", None)

            # Skip if text is empty or rating is missing / out of range
            if not text.strip():
                continue
            if rating is None or int(rating) not in SENTIMENT_MAP:
                continue

            records.append({
                "review_text": text.strip(),
                "rating":      int(rating),
            })

            # Stop early once we have enough valid records
            if len(records) >= n_samples * 3:   # over-sample then downsample
                break

    df = pd.DataFrame(records)

    # Stratified downsample to n_samples to keep rating balance
    if len(df) > n_samples:
        df = df.groupby("rating", group_keys=False).apply(
            lambda g: g.sample(frac=n_samples / len(df), random_state=SEED)
        ).reset_index(drop=True)
        # Guarantee we don't exceed n_samples
        df = df.sample(n=min(n_samples, len(df)), random_state=SEED).reset_index(drop=True)

    df["sentiment"] = df["rating"].map(SENTIMENT_MAP)
    df["category"]  = category_name

    print(f"  â†’ Loaded {len(df):,} reviews for '{category_name}'")
    return df

In [4]:
# Load all three categories
dfs = []
for cat_name, cat_path in CATEGORIES.items():
    df_cat = load_category(cat_path, SAMPLES_PER_CATEGORY, cat_name)
    dfs.append(df_cat)

# Combine into one DataFrame
data = pd.concat(dfs, ignore_index=True)
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle

print(f"\nTotal dataset size: {len(data):,} reviews")
print(f"Columns: {list(data.columns)}")

Loading 'sports' from Dataset\sports.json.gz ...



  Reading sports: 0 lines [00:00, ? lines/s]


  Reading sports: 6667 lines [00:00, 66638.29 lines/s]


  Reading sports: 13331 lines [00:00, 60693.09 lines/s]


  Reading sports: 19605 lines [00:00, 61576.94 lines/s]


  Reading sports: 26527 lines [00:00, 64498.46 lines/s]


  Reading sports: 33996 lines [00:00, 68097.20 lines/s]


  Reading sports: 36023 lines [00:00, 66036.98 lines/s]

KeyError: 'rating'

In [ ]:
# â”€â”€ Dataset statistics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Total samples  : {len(data):,}")
print()

print("Samples per category:")
print(data["category"].value_counts().to_string())
print()

print("Rating distribution:")
print(data["rating"].value_counts().sort_index().to_string())
print()

print("Sentiment distribution:")
sent_counts = data["sentiment"].value_counts().sort_index()
for idx, count in sent_counts.items():
    label = SENTIMENT_LABELS[idx]
    pct   = 100 * count / len(data)
    print(f"  {label:10s} ({idx}): {count:,}  ({pct:.1f}%)")

In [ ]:
# â”€â”€ Visualize rating distribution â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Rating distribution
rating_counts = data["rating"].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values,
            color=["#d62728", "#ff7f0e", "#bcbd22", "#2ca02c", "#1f77b4"],
            edgecolor="white")
axes[0].set_title("Rating Distribution (1â€“5 stars)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Star Rating")
axes[0].set_ylabel("Count")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(axes[0].patches, rating_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 100,
                 f"{val:,}", ha="center", va="bottom", fontsize=9)

# Sentiment distribution
sent_counts = data["sentiment"].value_counts().sort_index()
colors = ["#d62728", "#bcbd22", "#2ca02c"]
axes[1].bar([SENTIMENT_LABELS[i] for i in sent_counts.index],
            sent_counts.values, color=colors, edgecolor="white")
axes[1].set_title("Sentiment Class Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("Count")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(axes[1].patches, sent_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 100,
                 f"{val:,}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "dataset_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved.")

---
## Section 2: Derived Feature Definition

As required by Part A, we define a second task beyond sentiment classification. We choose **Review Length Category** as the derived feature.

**Motivation:** Review length correlates meaningfully with reviewer engagement. Very short reviews (few words) tend to be uninformative (e.g., "Great!"). Medium reviews often provide balanced feedback. Long reviews tend to be detailed and opinionated. Length is directly derivable from text alone â€” no external signals needed â€” making it a valid multi-task learning target. The model jointly learning to estimate length category should also encourage it to learn global structural properties of the review text.

**Label definition:**

| Class | Label | Word Count Range |
|---|---|---|
| 0 | Short  | 1â€“30 words   |
| 1 | Medium | 31â€“100 words |
| 2 | Long   | 101+ words   |

In [ ]:
# â”€â”€ Derived feature: Review length category â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
LENGTH_LABELS = ["Short (1-30)", "Medium (31-100)", "Long (101+)"]

def assign_length_category(text: str) -> int:
    """
    Assign a 3-class length label based on word count.
      0 = Short  : 1â€“30 words
      1 = Medium : 31â€“100 words
      2 = Long   : 101+ words
    """
    n_words = len(text.split())
    if n_words <= 30:
        return 0
    elif n_words <= 100:
        return 1
    else:
        return 2

data["length_label"] = data["review_text"].apply(assign_length_category)

print("Derived feature (review length category) distribution:")
for idx, label in enumerate(LENGTH_LABELS):
    count = (data["length_label"] == idx).sum()
    pct   = 100 * count / len(data)
    print(f"  {label:20s}: {count:,}  ({pct:.1f}%)")

---
## Section 3: Train / Validation / Test Split

We split **before** building the vocabulary to prevent data leakage. The vocabulary is constructed exclusively from the training set in Section 4.

In [ ]:
# Stratified split by sentiment label to preserve class balance across splits
# Step 1: Train vs Rest (85%)
train_df, temp_df = train_test_split(
    data,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=data["sentiment"],
    random_state=SEED
)

# Step 2: Val vs Test (50/50 of the remaining 30%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["sentiment"],
    random_state=SEED
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Dataset split summary:")
print(f"  Train : {len(train_df):,}  ({100*len(train_df)/len(data):.1f}%)")
print(f"  Val   : {len(val_df):,}   ({100*len(val_df)/len(data):.1f}%)")
print(f"  Test  : {len(test_df):,}   ({100*len(test_df)/len(data):.1f}%)")

# Confirm class balance is preserved
print("\nSentiment distribution across splits:")
for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    dist = split_df["sentiment"].value_counts(normalize=True).sort_index()
    dist_str = ", ".join([f"{SENTIMENT_LABELS[i]}: {v:.2%}" for i, v in dist.items()])
    print(f"  {split_name:6s}: {dist_str}")

---
## Section 4: Preprocessing Pipeline

### 4.1 Text Cleaning

We apply the following cleaning steps:
1. **Lowercase** â€” reduces vocabulary size
2. **HTML tag removal** â€” removes `<br/>`, `<p>`, etc.
3. **URL removal** â€” removes `http://...` patterns
4. **Punctuation normalization** â€” strip non-alphanumeric characters (keep apostrophes for contractions)
5. **Whitespace normalization** â€” collapse multiple spaces

In [ ]:
# â”€â”€ Compiled regex patterns (compile once for efficiency) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_RE_HTML    = re.compile(r"<[^>]+>")
_RE_URL     = re.compile(r"https?://\S+|www\.\S+")
_RE_NONWORD = re.compile(r"[^a-z0-9'\s]")
_RE_SPACES  = re.compile(r"\s+")
_RE_APOSTROPHE = re.compile(r"'s|n't|'re|'ve|'ll|'d|'m")  # common contractions

def clean_text(text: str) -> str:
    """
    Apply the full text cleaning pipeline:
      1. Lowercase
      2. Remove HTML tags
      3. Remove URLs
      4. Remove non-alphanumeric characters (keep apostrophes)
      5. Normalize whitespace

    Returns the cleaned string (never empty â€” returns a single space for
    completely empty reviews so downstream code does not crash).
    """
    text = text.lower()
    text = _RE_HTML.sub(" ", text)
    text = _RE_URL.sub(" ", text)
    text = _RE_NONWORD.sub(" ", text)
    text = _RE_SPACES.sub(" ", text).strip()
    return text if text else " "

# Quick test
sample = "This is <b>GREAT</b>! Check http://example.com for deals. Wasn't worth $50."
print("Before:", sample)
print("After :", clean_text(sample))

### 4.2 Tokenization

We use **simple whitespace tokenization** (word-level). After cleaning, the text contains only lowercase alphanumerics and apostrophes separated by spaces, so splitting on whitespace produces clean word tokens.

In [ ]:
def tokenize(text: str) -> list:
    """
    Tokenize a cleaned text string by splitting on whitespace.
    Returns a list of string tokens.
    """
    return text.split()

# Test
sample_clean = clean_text("This is GREAT! Wasn't worth $50.")
print("Tokens:", tokenize(sample_clean))

### 4.3 Vocabulary Construction

Built from **training data only** to prevent leakage. Tokens appearing fewer than `MIN_FREQ` times are excluded and mapped to `<UNK>` at inference time.

In [ ]:
class Vocabulary:
    """
    Word-level vocabulary built from a list of texts.

    Special tokens are always at indices 0-3:
      0: <PAD>  (padding)
      1: <UNK>  (unknown words)
      2: <SOS>  (start of sequence â€” used by decoder)
      3: <EOS>  (end of sequence   â€” used by decoder)
    """

    def __init__(self, min_freq: int = 2):
        self.min_freq = min_freq
        self.token2idx: dict = {}
        self.idx2token: dict = {}
        self.token_freq: Counter = Counter()

        # Reserve special token indices
        for i, tok in enumerate(SPECIAL_TOKENS):
            self.token2idx[tok] = i
            self.idx2token[i]   = tok

    @property
    def pad_idx(self): return self.token2idx[PAD_TOKEN]

    @property
    def unk_idx(self): return self.token2idx[UNK_TOKEN]

    @property
    def sos_idx(self): return self.token2idx[SOS_TOKEN]

    @property
    def eos_idx(self): return self.token2idx[EOS_TOKEN]

    def __len__(self):
        return len(self.token2idx)

    def build(self, texts: list):
        """
        Build vocabulary from a list of raw text strings.
        Applies cleaning + tokenization internally.
        Only tokens with freq >= min_freq are added.
        """
        print("Building vocabulary from training data...")
        for text in tqdm(texts, desc="  Counting tokens"):
            tokens = tokenize(clean_text(text))
            self.token_freq.update(tokens)

        next_idx = len(SPECIAL_TOKENS)  # start after reserved special tokens
        added = 0
        for token, freq in sorted(self.token_freq.items()):
            if freq >= self.min_freq and token not in self.token2idx:
                self.token2idx[token] = next_idx
                self.idx2token[next_idx] = token
                next_idx += 1
                added += 1

        print(f"  Unique tokens in training : {len(self.token_freq):,}")
        print(f"  Tokens added (freq>={self.min_freq:d})    : {added:,}")
        print(f"  Total vocabulary size     : {len(self):,}")
        return self

    def numericalize(self, text: str, add_special: bool = False) -> list:
        """
        Convert a raw text string to a list of integer indices.
        Unknown tokens map to unk_idx.
        If add_special=True, prepends SOS and appends EOS.
        """
        tokens = tokenize(clean_text(text))
        indices = [self.token2idx.get(t, self.unk_idx) for t in tokens]
        if add_special:
            indices = [self.sos_idx] + indices + [self.eos_idx]
        return indices

    def decode(self, indices: list) -> str:
        """Convert a list of integer indices back to a token string."""
        return " ".join(self.idx2token.get(i, UNK_TOKEN) for i in indices)

    def save(self, path: Path):
        with open(path, "wb") as f:
            pickle.dump(self, f)
        print(f"Vocabulary saved to {path}")

    @classmethod
    def load(cls, path: Path):
        with open(path, "rb") as f:
            vocab = pickle.load(f)
        print(f"Vocabulary loaded from {path} (size={len(vocab):,})")
        return vocab

In [ ]:
# Build vocabulary from TRAINING data only
vocab = Vocabulary(min_freq=MIN_FREQ)
vocab.build(train_df["review_text"].tolist())

# Save vocabulary for reuse in later chunks
vocab.save(RESULTS_DIR / "vocabulary.pkl")

In [ ]:
# Sanity checks
assert vocab.pad_idx == 0, "PAD should be index 0"
assert vocab.unk_idx == 1, "UNK should be index 1"
assert vocab.sos_idx == 2, "SOS should be index 2"
assert vocab.eos_idx == 3, "EOS should be index 3"

sample_text = train_df["review_text"].iloc[0]
sample_ids  = vocab.numericalize(sample_text)
print(f"Sample review (first 50 chars): '{sample_text[:50]}...'")
print(f"Numericalized (first 15 ids)  : {sample_ids[:15]}")
print(f"Decoded back (first 15 tokens): {vocab.decode(sample_ids[:15])}")

### 4.4 Padding & Truncation

All sequences are padded (with `<PAD>`) or truncated to exactly `MAX_SEQ_LEN` tokens. Truncation removes tokens from the **end** of the sequence.

In [ ]:
def pad_or_truncate(indices: list, max_len: int, pad_idx: int) -> list:
    """
    Truncate or pad a list of indices to exactly `max_len`.

    - Truncation: removes tokens from the right (end of sequence)
    - Padding:    appends pad_idx tokens on the right

    Returns a list of length exactly max_len.
    """
    if len(indices) >= max_len:
        return indices[:max_len]
    return indices + [pad_idx] * (max_len - len(indices))


def preprocess_text(text: str, vocab: Vocabulary, max_len: int,
                    add_special: bool = False) -> list:
    """
    End-to-end preprocessing for a single review text:
      clean â†’ tokenize â†’ numericalize â†’ pad/truncate

    Args:
        text        : raw review string
        vocab       : built Vocabulary object
        max_len     : fixed output length
        add_special : if True, prepend SOS and append EOS before padding

    Returns:
        A list of integer token indices of length exactly max_len.
    """
    indices = vocab.numericalize(text, add_special=add_special)
    return pad_or_truncate(indices, max_len, vocab.pad_idx)


# Verify
test_ids = preprocess_text(train_df["review_text"].iloc[0], vocab, MAX_SEQ_LEN)
assert len(test_ids) == MAX_SEQ_LEN, f"Expected {MAX_SEQ_LEN}, got {len(test_ids)}"
print(f"Preprocessed sequence length: {len(test_ids)} âœ“")
print(f"First 10 indices: {test_ids[:10]}")
print(f"Last  10 indices: {test_ids[-10:]}  (trailing PADs = {test_ids.count(vocab.pad_idx)} total)")

---
## Section 5: Preprocess & Save All Splits

We preprocess the full train, validation, and test sets and save them as NumPy arrays. This avoids re-running preprocessing every time later chunks are executed.

In [ ]:
def preprocess_split(df: pd.DataFrame,
                     vocab: Vocabulary,
                     max_len: int,
                     split_name: str,
                     add_special: bool = False) -> dict:
    """
    Preprocess an entire DataFrame split.

    Returns a dict with:
      'input_ids'    : np.ndarray (N, max_len) of int32 â€” token indices
      'sentiment'    : np.ndarray (N,)          of int64 â€” sentiment labels (0/1/2)
      'length_label' : np.ndarray (N,)          of int64 â€” derived feature labels
      'rating'       : np.ndarray (N,)          of int64 â€” original star ratings
    """
    print(f"Preprocessing {split_name} split ({len(df):,} samples)...")
    input_ids = np.array(
        [
            preprocess_text(row, vocab, max_len, add_special=add_special)
            for row in tqdm(df["review_text"], desc=f"  {split_name}")
        ],
        dtype=np.int32
    )
    return {
        "input_ids":    input_ids,
        "sentiment":    df["sentiment"].values.astype(np.int64),
        "length_label": df["length_label"].values.astype(np.int64),
        "rating":       df["rating"].values.astype(np.int64),
    }


# Preprocess all splits
train_data = preprocess_split(train_df, vocab, MAX_SEQ_LEN, "Train")
val_data   = preprocess_split(val_df,   vocab, MAX_SEQ_LEN, "Val")
test_data  = preprocess_split(test_df,  vocab, MAX_SEQ_LEN, "Test")

print("\nShapes:")
print(f"  train input_ids : {train_data['input_ids'].shape}")
print(f"  val   input_ids : {val_data['input_ids'].shape}")
print(f"  test  input_ids : {test_data['input_ids'].shape}")

In [ ]:
# â”€â”€ Save preprocessed arrays â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
for split_name, split_data in [("train", train_data),
                                ("val",   val_data),
                                ("test",  test_data)]:
    for array_name, array in split_data.items():
        path = RESULTS_DIR / f"{split_name}_{array_name}.npy"
        np.save(path, array)

print("All preprocessed arrays saved to results/.")
print("\nSaved files:")
for f in sorted(RESULTS_DIR.glob("*.npy")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s}  {size_mb:.1f} MB")

---
## Section 6: Preprocessing Analysis & Visualizations

In [ ]:
# â”€â”€ Review length analysis (before and after truncation) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
raw_lengths   = train_df["review_text"].apply(lambda t: len(tokenize(clean_text(t))))
clipped_lengths = np.minimum(raw_lengths.values, MAX_SEQ_LEN)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(raw_lengths, bins=60, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(MAX_SEQ_LEN, color="red", linestyle="--", linewidth=1.5,
                label=f"MAX_SEQ_LEN = {MAX_SEQ_LEN}")
axes[0].set_title("Raw Token Length Distribution (Training)", fontweight="bold")
axes[0].set_xlabel("Number of Tokens")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(clipped_lengths, bins=60, color="darkorange", edgecolor="white", alpha=0.85)
axes[1].set_title("Token Length After Truncation", fontweight="bold")
axes[1].set_xlabel("Number of Tokens (capped at MAX_SEQ_LEN)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "token_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

pct_truncated = (raw_lengths > MAX_SEQ_LEN).mean() * 100
pct_padded    = (raw_lengths < MAX_SEQ_LEN).mean() * 100
print(f"Truncated reviews : {pct_truncated:.1f}%")
print(f"Padded reviews    : {pct_padded:.1f}%")
print(f"Mean raw length   : {raw_lengths.mean():.1f} tokens")
print(f"Median raw length : {raw_lengths.median():.1f} tokens")
print(f"Max raw length    : {raw_lengths.max()} tokens")

In [ ]:
# â”€â”€ Vocabulary coverage on val and test sets â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def compute_coverage(df, vocab):
    """What fraction of tokens in df appear in vocab?"""
    total, known = 0, 0
    for text in df["review_text"]:
        tokens = tokenize(clean_text(text))
        for t in tokens:
            total += 1
            if t in vocab.token2idx:
                known += 1
    return known / total if total else 0

train_cov = compute_coverage(train_df, vocab)
val_cov   = compute_coverage(val_df,   vocab)
test_cov  = compute_coverage(test_df,  vocab)

print("Vocabulary coverage (fraction of tokens in vocab):")
print(f"  Train : {train_cov:.4f}  ({100*train_cov:.2f}%)")
print(f"  Val   : {val_cov:.4f}  ({100*val_cov:.2f}%)")
print(f"  Test  : {test_cov:.4f}  ({100*test_cov:.2f}%)")

In [ ]:
# â”€â”€ Top 30 most frequent tokens (excluding special tokens) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
top_tokens = [
    (tok, freq)
    for tok, freq in vocab.token_freq.most_common()
    if tok not in SPECIAL_TOKENS
][:30]

toks, freqs = zip(*top_tokens)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(toks)), freqs, color="slateblue", edgecolor="white")
ax.set_xticks(range(len(toks)))
ax.set_xticklabels(toks, rotation=45, ha="right", fontsize=9)
ax.set_title("Top 30 Most Frequent Tokens (Training Set)", fontweight="bold")
ax.set_ylabel("Frequency")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(PLOTS_DIR / "top_tokens.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 7: Preprocessing Summary

Print a complete summary of all preprocessing decisions for the report.

In [ ]:
print("=" * 60)
print("PREPROCESSING PIPELINE SUMMARY")
print("=" * 60)
print(f"Dataset categories   : {list(CATEGORIES.keys())}")
print(f"Samples per category : ~{SAMPLES_PER_CATEGORY:,}")
print(f"Total samples        : {len(data):,}")
print()
print("Splits:")
print(f"  Train : {len(train_df):,}  ({100*len(train_df)/len(data):.1f}%)")
print(f"  Val   : {len(val_df):,}    ({100*len(val_df)/len(data):.1f}%)")
print(f"  Test  : {len(test_df):,}    ({100*len(test_df)/len(data):.1f}%)")
print()
print("Cleaning steps:")
print("  1. Lowercase")
print("  2. HTML tag removal")
print("  3. URL removal")
print("  4. Non-alphanumeric character removal (apostrophes kept)")
print("  5. Whitespace normalization")
print()
print("Tokenization : whitespace split on cleaned text")
print(f"Vocabulary   : {len(vocab):,} tokens (min_freq={MIN_FREQ})")
print(f"Special toks : {SPECIAL_TOKENS}")
print(f"Max seq len  : {MAX_SEQ_LEN} tokens")
print(f"Truncated    : {pct_truncated:.1f}% of training reviews")
print(f"Padded       : {pct_padded:.1f}% of training reviews")
print()
print("Tasks:")
print("  Task 1 (Sentiment) : 3-class (Negative/Neutral/Positive)")
print("                       ratings 1-2â†’0, 3â†’1, 4-5â†’2")
print("  Task 2 (Derived)   : Review length category")
print("                       Short(0)=1-30 words, Medium(1)=31-100, Long(2)=101+")
print()
print("Saved artifacts:")
for f in sorted(RESULTS_DIR.glob("*")):
    if f.is_file():
        print(f"  {f}")
print("=" * 60)
print("Chunk 1 complete. Ready for Chunk 2 (Encoder Architecture).")

---
## Chunk 2: Encoder Architecture from Scratch

In this section, we implement the core components of the Transformer encoder using only primitive PyTorch operations. We strictly avoid `nn.Transformer` and `nn.MultiheadAttention` as per the assignment restrictions.

### Section 7: Positional Encoding

Since the Transformer architecture does not have recurrence or convolution, it has no inherent sense of token order. We inject positional information using sine and cosine functions of different frequencies.

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    """
    Standard Sinusoidal Positional Encoding.
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Compute the positional encodings once in log space
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Shape: [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input embeddings [batch_size, seq_len, d_model]
        Returns:
            Embeddings + Positional Encoding
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

### Section 8: Multi-Head Attention (MHA)

MHA allows the model to jointly attend to information from different representation subspaces at different positions.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear layers for Q, K, V projections
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        
        self.out_linear = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 1. Linear projections and split into heads
        # [batch, seq_len, d_model] -> [batch, seq_len, heads, d_k] -> [batch, heads, seq_len, d_k]
        Q = self.q_linear(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.k_linear(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.v_linear(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. Scaled Dot-Product Attention
        # scores: [batch, heads, seq_len_q, seq_len_k]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            # mask: [batch, 1, 1, seq_len_k] or [batch, 1, seq_len_q, seq_len_k]
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # 3. Context vector
        # [batch, heads, seq_len_q, d_k]
        context = torch.matmul(attn_weights, V)
        
        # 4. Concatenate heads and project
        # [batch, seq_len_q, d_model]
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.out_linear(context)

### Section 9: Feed-Forward Network (FFN)

Each encoder block contains a position-wise fully connected feed-forward network.

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()  # GELU is often preferred in modern Transformers

    def forward(self, x):
        return self.w_2(self.dropout(self.activation(self.w_1(x))))

### Section 10: Encoder Block

Each block consists of two sub-layers: a multi-head self-attention mechanism and a position-wise feed-forward network. We use residual connections around each sub-layer, followed by layer normalization.

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionWiseFeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 1. Multi-Head Self-Attention + Residual + Norm
        attn_output = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # 2. Feed-Forward + Residual + Norm
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        return x

### Section 11: Transformer Encoder

The full encoder is a stack of $N$ encoder blocks.

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_layers: int, 
                 num_heads: int, d_ff: int, max_len: int, dropout: float = 0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # x: [batch_size, seq_len]
        x = self.embedding(x)  # [batch_size, seq_len, d_model]
        x = self.pos_encoding(x)
        
        for layer in self.layers:
            x = layer(x, mask)
            
        return self.norm(x)

### Section 12: Sanity Check

We verify the encoder architecture by passing a dummy batch through it.

In [ ]:
# Hyperparameters for sanity check
V_SIZE = 1000
D_MOD  = 128
N_LAY  = 2
N_HEAD = 4
D_FF   = 512
M_LEN  = 128

test_encoder = TransformerEncoder(V_SIZE, D_MOD, N_LAY, N_HEAD, D_FF, M_LEN)
dummy_input = torch.randint(0, V_SIZE, (8, M_LEN))  # [batch=8, seq=128]
dummy_output = test_encoder(dummy_input)

print(f'Input shape  : {dummy_input.shape}')
print(f'Output shape : {dummy_output.shape}  (Expected: [8, 128, 128])')
assert dummy_output.shape == (8, M_LEN, D_MOD)
print('Encoder sanity check passed! \u2713')

---
## Chunk 3: Multi-Task Training Pipeline

We wrap the encoder into a multi-task model and train it end-to-end on two objectives:
1. **Sentiment classification** (3 classes: Negative / Neutral / Positive)
2. **Review length category** (3 classes: Short / Medium / Long)

A combined cross-entropy loss is used:  `L = α·L_sentiment + β·L_length`

### Section 13: Encoder Hyperparameters


In [ ]:
# ── Model hyperparameters ─────────────────────────────────────────────────────
D_MODEL    = 256      # hidden dimension
N_LAYERS   = 4        # number of encoder blocks
N_HEADS    = 8        # attention heads (must divide D_MODEL evenly)
D_FF       = 1024     # feed-forward inner dimension
DROPOUT    = 0.1
BATCH_SIZE = 64
EPOCHS     = 10
LR         = 1e-4
ALPHA      = 1.0      # weight for sentiment loss
BETA       = 0.5      # weight for length loss

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'd_model={D_MODEL}, n_layers={N_LAYERS}, n_heads={N_HEADS}, d_ff={D_FF}')

### Section 14: Multi-Task Encoder Model

Shared encoder trunk with two separate linear classification heads. We apply **global average pooling** over the sequence dimension to obtain a fixed-size review representation.

In [ ]:
class MultiTaskEncoderModel(nn.Module):
    """
    Encoder-only multi-task model.

    Architecture:
      Embedding → Positional Encoding → N × EncoderBlock → Global Avg Pool
      ├─ Linear → 3 sentiment logits
      └─ Linear → 3 length logits
    """
    def __init__(self, encoder: TransformerEncoder, d_model: int,
                 n_sentiment: int = 3, n_length: int = 3):
        super().__init__()
        self.encoder        = encoder
        self.sentiment_head = nn.Linear(d_model, n_sentiment)
        self.length_head    = nn.Linear(d_model, n_length)

    def forward(self, x, mask=None):
        # x: [B, S]
        enc_out = self.encoder(x, mask)   # [B, S, D]

        # Pool over sequence dimension
        if mask is not None:
            # Mask: [B,1,1,S] → squeeze to [B,S,1] to zero pad positions
            token_mask = mask.squeeze(1).squeeze(1).unsqueeze(-1).float()  # [B,S,1]
            pooled = (enc_out * token_mask).sum(dim=1) / token_mask.sum(dim=1).clamp(min=1)
        else:
            pooled = enc_out.mean(dim=1)  # [B, D]

        return self.sentiment_head(pooled), self.length_head(pooled)

### Section 15: Dataset & DataLoaders


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ReviewDataset(Dataset):
    """Loads preprocessed .npy arrays for a given split."""
    def __init__(self, split: str):
        self.ids  = np.load(RESULTS_DIR / f'{split}_input_ids.npy')
        self.sent = np.load(RESULTS_DIR / f'{split}_sentiment.npy')
        self.llen = np.load(RESULTS_DIR / f'{split}_length_label.npy')

    def __len__(self):  return len(self.ids)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.ids[idx],  dtype=torch.long),
            torch.tensor(self.sent[idx], dtype=torch.long),
            torch.tensor(self.llen[idx], dtype=torch.long),
        )

train_loader = DataLoader(ReviewDataset('train'), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(ReviewDataset('val'),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(ReviewDataset('test'),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

### Section 16: Training & Evaluation Functions

Combined loss = **α × CrossEntropy(sentiment) + β × CrossEntropy(length)**

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

CRITERION = nn.CrossEntropyLoss()

def make_pad_mask(ids, pad_idx):
    """Build additive attention mask: [B, 1, 1, S] — 0 = real, 1 = pad."""
    return (ids != pad_idx).unsqueeze(1).unsqueeze(2)   # True for real tokens


def run_epoch(model, loader, optimizer, device, alpha, beta, training):
    model.train() if training else model.eval()
    total_loss = 0.0
    s_preds, s_labels, l_preds, l_labels = [], [], [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for ids, sent, llen in loader:
            ids, sent, llen = ids.to(device), sent.to(device), llen.to(device)
            mask = make_pad_mask(ids, vocab.pad_idx).to(device)

            s_logit, l_logit = model(ids, mask)
            loss = alpha * CRITERION(s_logit, sent) + beta * CRITERION(l_logit, llen)

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            s_preds.extend(s_logit.argmax(-1).cpu().tolist())
            s_labels.extend(sent.cpu().tolist())
            l_preds.extend(l_logit.argmax(-1).cpu().tolist())
            l_labels.extend(llen.cpu().tolist())

    avg_loss  = total_loss / len(loader)
    sent_acc  = accuracy_score(s_labels, s_preds)
    sent_f1   = f1_score(s_labels, s_preds, average='weighted', zero_division=0)
    sent_pre  = precision_score(s_labels, s_preds, average='weighted', zero_division=0)
    sent_rec  = recall_score(s_labels, s_preds, average='weighted', zero_division=0)
    len_acc   = accuracy_score(l_labels, l_preds)
    return avg_loss, sent_acc, sent_f1, sent_pre, sent_rec, len_acc

### Section 17: Model Initialisation & Training


In [ ]:
# Build model
encoder_net = TransformerEncoder(
    vocab_size=len(vocab), d_model=D_MODEL, n_layers=N_LAYERS,
    num_heads=N_HEADS, d_ff=D_FF, max_len=MAX_SEQ_LEN, dropout=DROPOUT
)
model     = MultiTaskEncoderModel(encoder_net, D_MODEL).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

# ── Training history ──────────────────────────────────────────────────────────
history = {k: [] for k in ('train_loss','val_loss','sent_acc','sent_f1','sent_pre','sent_rec','len_acc')}

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    tr_loss, *_ = run_epoch(model, train_loader, optimizer, DEVICE, ALPHA, BETA, training=True)
    vl_loss, v_sacc, v_sf1, v_spre, v_srec, v_lacc = run_epoch(
        model, val_loader, optimizer, DEVICE, ALPHA, BETA, training=False)

    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['sent_acc'].append(v_sacc)
    history['sent_f1'].append(v_sf1)
    history['sent_pre'].append(v_spre)
    history['sent_rec'].append(v_srec)
    history['len_acc'].append(v_lacc)

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), MODELS_DIR / 'encoder_model.pt')

    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'Train Loss: {tr_loss:.4f} | Val Loss: {vl_loss:.4f} | '
          f'Sent Acc: {v_sacc:.2%} | Sent F1: {v_sf1:.4f} | Len Acc: {v_lacc:.2%}')

print(f'\nBest val loss: {best_val_loss:.4f} — weights saved to {MODELS_DIR}/encoder_model.pt')

### Section 18: Learning Curves


In [ ]:
epochs_x = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs_x, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs_x, history['val_loss'],   'r-o', label='Val')
axes[0].set_title('Loss', fontweight='bold'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Sentiment metrics
axes[1].plot(epochs_x, history['sent_acc'], 'g-o', label='Accuracy')
axes[1].plot(epochs_x, history['sent_f1'],  'b-s', label='F1 (weighted)')
axes[1].plot(epochs_x, history['sent_pre'], 'm-^', label='Precision')
axes[1].plot(epochs_x, history['sent_rec'], 'c-v', label='Recall')
axes[1].set_title('Sentiment Metrics', fontweight='bold'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Length accuracy
axes[2].plot(epochs_x, history['len_acc'], 'darkorange', marker='D', label='Length Acc')
axes[2].set_title('Length Category Accuracy', fontweight='bold'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Learning curves saved.')

### Section 19: Final Evaluation on Test Set


In [ ]:
import json as _json

# Load best checkpoint
model.load_state_dict(torch.load(MODELS_DIR / 'encoder_model.pt', map_location=DEVICE))

_, test_sacc, test_sf1, test_spre, test_srec, test_lacc = run_epoch(
    model, test_loader, None, DEVICE, ALPHA, BETA, training=False)

encoder_metrics = {
    'sentiment': {
        'accuracy':  round(test_sacc, 4),
        'f1':        round(test_sf1,  4),
        'precision': round(test_spre, 4),
        'recall':    round(test_srec, 4),
    },
    'length_category': {
        'accuracy':  round(test_lacc, 4),
    }
}

with open(RESULTS_DIR / 'encoder_metrics.json', 'w') as f:
    _json.dump(encoder_metrics, f, indent=2)

print('=== Test Set Results ===')
print(f'  Sentiment  Acc : {test_sacc:.2%}')
print(f'  Sentiment  F1  : {test_sf1:.4f}')
print(f'  Sentiment  Pre : {test_spre:.4f}')
print(f'  Sentiment  Rec : {test_srec:.4f}')
print(f'  Length Cat Acc : {test_lacc:.2%}')
print(f'Results saved to {RESULTS_DIR}/encoder_metrics.json')

---
## Chunk 4: Save Embeddings + Retrieval Module

We extract encoder embeddings for the entire training corpus and store them as NumPy arrays. A cosine-similarity retrieval module then finds the top-k most similar training reviews for any test query — these retrieved reviews serve as context for the decoder.

### Section 20: Extract and Save Training Embeddings

We load the best encoder checkpoint and pass every training sample through it (no gradients) to obtain a fixed-dimensional representation via global average pooling.

In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / 'encoder_model.pt', map_location=DEVICE))
model.eval()

all_embeddings = []
all_sent_preds = []
all_len_preds  = []

with torch.no_grad():
    for ids, sent, llen in tqdm(train_loader, desc='Extracting embeddings'):
        ids  = ids.to(DEVICE)
        mask = make_pad_mask(ids, vocab.pad_idx).to(DEVICE)

        # Get raw encoder output  [B, S, D]
        enc_out = model.encoder(ids, mask)

        # Masked global average pool  [B, D]
        token_mask = mask.squeeze(1).squeeze(1).unsqueeze(-1).float()
        pooled = (enc_out * token_mask).sum(1) / token_mask.sum(1).clamp(min=1)

        s_logit, l_logit = model.sentiment_head(pooled), model.length_head(pooled)

        all_embeddings.append(pooled.cpu().numpy())
        all_sent_preds.extend(s_logit.argmax(-1).cpu().tolist())
        all_len_preds.extend(l_logit.argmax(-1).cpu().tolist())

train_embeddings = np.concatenate(all_embeddings, axis=0)   # [N_train, D_MODEL]
train_labels     = np.load(RESULTS_DIR / 'train_sentiment.npy')

np.save(RESULTS_DIR / 'train_embeddings.npy', train_embeddings)
np.save(RESULTS_DIR / 'train_labels.npy',     train_labels)

print(f'Embeddings shape : {train_embeddings.shape}')
print(f'Labels shape     : {train_labels.shape}')
print('Saved to results/')

### Section 21: Retrieval Module

Given a query embedding (from a test review), we rank all training embeddings by **cosine similarity** and return the top-k most similar reviews together with their labels.

In [ ]:
class RetrievalModule:
    """
    Stores training embeddings and performs cosine-similarity top-k retrieval.

    Args:
        embeddings  : np.ndarray [N, D]  — L2-normalised training embeddings
        texts       : list[str]          — corresponding raw review texts
        labels      : np.ndarray [N]     — sentiment labels (0/1/2)
    """
    def __init__(self, embeddings: np.ndarray, texts: list, labels: np.ndarray):
        # L2-normalise once so cosine sim = dot product
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        self.embeddings = embeddings / np.clip(norms, 1e-9, None)   # [N, D]
        self.texts  = texts
        self.labels = labels

    def retrieve(self, query_vec: np.ndarray, k: int = 3):
        """
        Args:
            query_vec : np.ndarray [D]  — a single (optionally unnormalised) query vector
            k         : int             — number of results to return

        Returns:
            List of dicts with keys: rank, text, label, similarity
        """
        q = query_vec / max(np.linalg.norm(query_vec), 1e-9)
        sims   = self.embeddings @ q               # [N]
        top_k  = np.argsort(sims)[::-1][:k]
        return [
            dict(rank=i+1, text=self.texts[idx],
                 label=SENTIMENT_LABELS[int(self.labels[idx])],
                 similarity=float(sims[idx]))
            for i, idx in enumerate(top_k)
        ]

In [ ]:
# Load saved embeddings
train_emb_np  = np.load(RESULTS_DIR / 'train_embeddings.npy')
train_lbl_np  = np.load(RESULTS_DIR / 'train_labels.npy')
train_texts   = train_df['review_text'].tolist()

retriever = RetrievalModule(train_emb_np, train_texts, train_lbl_np)
print(f'Retriever ready — index size: {len(train_texts):,} reviews')

### Section 22: Retrieval Quality Analysis

We run three example queries from the test set and inspect the returned matches to assess whether the retrieval is semantically meaningful.

In [ ]:
def embed_text(raw_text: str, model, vocab, device):
    """Convert a raw review string to a pooled embedding vector."""
    ids = torch.tensor(
        [preprocess_text(raw_text, vocab, MAX_SEQ_LEN)], dtype=torch.long
    ).to(device)
    mask = make_pad_mask(ids, vocab.pad_idx).to(device)
    model.eval()
    with torch.no_grad():
        enc_out    = model.encoder(ids, mask)                          # [1, S, D]
        token_mask = mask.squeeze(1).squeeze(1).unsqueeze(-1).float()
        pooled     = (enc_out * token_mask).sum(1) / token_mask.sum(1).clamp(min=1)
    return pooled.squeeze(0).cpu().numpy()                             # [D]


# Pick 3 test examples with different sentiments
for sent_class in [0, 1, 2]:
    idx = test_df[test_df['sentiment'] == sent_class].index[0]
    query_text = test_df.loc[idx, 'review_text']
    true_label = SENTIMENT_LABELS[sent_class]

    q_vec   = embed_text(query_text, model, vocab, DEVICE)
    results = retriever.retrieve(q_vec, k=3)

    print('=' * 70)
    print(f'QUERY [{true_label}]: {query_text[:120]}...')
    print('-' * 70)
    for r in results:
        print(f"  Rank {r['rank']} (sim={r['similarity']:.4f}, label={r['label']}): "
              f"{r['text'][:100]}...")
    print()

### Section 23: Effect of Varying k

We examine how the similarity scores change as we expand k from 1 to 10.

In [ ]:
# Use first test review as the fixed query
q_text = test_df['review_text'].iloc[0]
q_vec  = embed_text(q_text, model, vocab, DEVICE)

k_values = [1, 3, 5, 10]
print(f'Query: {q_text[:100]}...')
print()
for k in k_values:
    top = retriever.retrieve(q_vec, k=k)
    sims = [r['similarity'] for r in top]
    print(f'k={k:2d}  | avg sim: {sum(sims)/len(sims):.4f} | '
          f'min sim: {min(sims):.4f} | max sim: {max(sims):.4f}')

---
## Part C: Decoder Model for Explanation Generation

We build a **decoder-only Transformer from scratch** that generates a 1–2 sentence natural language explanation for why a review carries its predicted sentiment.

**Input template:**
```
review: <text> sentiment: <label> feature: <length_label> context: <retrieved_1> ... <retrieved_k> explanation:
```

The decoder is trained with a **language-modeling (next-token prediction)** objective on synthetically constructed explanation targets built from the review text + labels.

### Section 24: Decoder Hyperparameters

In [ ]:
# ── Decoder hyperparameters ─────────────────────────────────────────────────
DEC_D_MODEL   = 256      # embedding / hidden dimension
DEC_NUM_HEADS = 4        # attention heads  (256 / 4 = 64 per head)
DEC_D_FF      = 512      # feed-forward inner dimension
DEC_NUM_LAYERS= 3        # stacked decoder blocks
DEC_DROPOUT   = 0.1
DEC_MAX_LEN   = 192      # max tokens for the full prompt + explanation
DEC_EPOCHS    = 5
DEC_LR        = 3e-4
DEC_BATCH     = 32
DEC_K_CONTEXT = 2        # number of retrieved reviews to include in prompt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Decoder config  d_model={DEC_D_MODEL}  heads={DEC_NUM_HEADS}  layers={DEC_NUM_LAYERS}')
print(f'Device: {DEVICE}')

### Section 25: Causal Masking

The decoder uses a **causal (auto-regressive) mask** so that position $i$ can only attend to positions $\leq i$.

In [ ]:
def make_causal_mask(seq_len: int, device) -> torch.Tensor:
    """
    Returns an upper-triangular boolean mask of shape (seq_len, seq_len).
    True  => position is masked (set to -inf before softmax).
    False => position is visible.
    """
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
    return mask   # shape: (T, T)

# Quick sanity check
_m = make_causal_mask(5, 'cpu')
print("Causal mask (True = blocked):")
print(_m.int())

### Section 26: Decoder Block

Each block contains:
1. **Masked Multi-Head Self-Attention** — causal mask prevents future-token leakage
2. **Layer Normalization + Residual** after attention
3. **Position-Wise Feed-Forward** sublayer
4. **Layer Normalization + Residual** after FFN

In [ ]:
class DecoderMHA(nn.Module):
    """Masked multi-head self-attention for the decoder."""

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def _split_heads(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x, causal_mask=None, pad_mask=None):
        B, T, D = x.shape
        Q = self._split_heads(self.W_q(x))   # (B, H, T, dk)
        K = self._split_heads(self.W_k(x))
        V = self._split_heads(self.W_v(x))

        scale = self.d_k ** -0.5
        scores = torch.matmul(Q, K.transpose(-2, -1)) * scale   # (B, H, T, T)

        if causal_mask is not None:
            # causal_mask: (T, T) bool
            scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        if pad_mask is not None:
            # pad_mask: (B, T) bool  — True where PAD
            scores = scores.masked_fill(pad_mask.unsqueeze(1).unsqueeze(2), float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        attn = self.drop(attn)
        ctx  = torch.matmul(attn, V)                            # (B, H, T, dk)
        ctx  = ctx.transpose(1, 2).contiguous().view(B, T, D)  # (B, T, D)
        return self.W_o(ctx)


class DecoderBlock(nn.Module):
    """Single decoder block: masked self-attention + FFN with pre-norm."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attn  = DecoderMHA(d_model, num_heads, dropout)
        self.ff    = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, causal_mask=None, pad_mask=None):
        # Sublayer 1: masked self-attention + residual
        x = x + self.drop(self.attn(self.norm1(x), causal_mask=causal_mask, pad_mask=pad_mask))
        # Sublayer 2: FFN + residual
        x = x + self.drop(self.ff(self.norm2(x)))
        return x


print("DecoderBlock and DecoderMHA defined.")

### Section 27: Decoder Transformer Model

In [ ]:
class DecoderTransformer(nn.Module):
    """
    Decoder-only Transformer for autoregressive explanation generation.

    Input : token index sequence of length T
    Output: logits over vocabulary of shape (B, T, vocab_size)
    """

    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 d_ff: int, num_layers: int, max_len: int, dropout: float = 0.1):
        super().__init__()
        self.d_model   = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(d_model, max_len, dropout)
        self.blocks    = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm_final = nn.LayerNorm(d_model)
        self.proj       = nn.Linear(d_model, vocab_size, bias=False)
        # Weight-tie token embedding and output projection
        self.proj.weight = self.token_emb.weight

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x, pad_mask=None):
        """
        x        : (B, T) long tensor of token indices
        pad_mask : (B, T) bool tensor — True where padding
        Returns logits (B, T, V).
        """
        T = x.size(1)
        causal_mask = make_causal_mask(T, x.device)   # (T, T)

        h = self.pos_enc(self.token_emb(x))            # (B, T, D)
        for block in self.blocks:
            h = block(h, causal_mask=causal_mask, pad_mask=pad_mask)
        h = self.norm_final(h)
        return self.proj(h)                             # (B, T, V)


print("DecoderTransformer defined.")

### Section 28: Decoder Sanity Check

In [ ]:
_dec_test = DecoderTransformer(
    vocab_size=len(vocab), d_model=DEC_D_MODEL,
    num_heads=DEC_NUM_HEADS, d_ff=DEC_D_FF,
    num_layers=DEC_NUM_LAYERS, max_len=DEC_MAX_LEN,
    dropout=0.0
).to(DEVICE)

_x = torch.randint(1, 100, (2, DEC_MAX_LEN)).to(DEVICE)
_logits = _dec_test(_x)
assert _logits.shape == (2, DEC_MAX_LEN, len(vocab)), f"Unexpected shape {_logits.shape}"
print(f"Decoder sanity check passed  output shape: {tuple(_logits.shape)}")
del _dec_test, _x, _logits

### Section 29: Build Training Sequences

We construct each training sequence using the template:
```
review: <text> sentiment: <label> feature: <length> context: <r1> <r2> explanation: <synthetic>
```
The **synthetic explanation** is a short factual sentence derived from the labels, giving the decoder real signal to learn from.


In [ ]:
def build_prompt(review_text: str,
                 sent_label: str,
                 feat_label: str,
                 context_texts: list,
                 explanation: str,
                 vocab,
                 max_len: int) -> list:
    """
    Construct and numericalize the full decoder input sequence.
    Returns a list of token indices padded/truncated to max_len.
    Template:
      review: <text> sentiment: <label> feature: <feat>
      context: <c1> <c2> explanation: <expl>
    """
    # Build raw string
    ctx_part = ' '.join(context_texts) if context_texts else 'none'
    prompt = (f"review {review_text} sentiment {sent_label} "
              f"feature {feat_label} context {ctx_part} "
              f"explanation {explanation}")

    ids = vocab.numericalize(prompt, add_special=False)
    ids = [vocab.sos_idx] + ids + [vocab.eos_idx]
    return pad_or_truncate(ids, max_len, vocab.pad_idx)


def make_synthetic_explanation(sent_label: str, feat_label: str, review_text: str) -> str:
    """Rule-based explanation target used for LM training."""
    snippet = ' '.join(review_text.split()[:12])
    return (f"the review is {sent_label} because it is a {feat_label} "
            f"review that starts with {snippet}")


print("Prompt builder and synthetic explanation generator defined.")
# Quick demo
_demo_expl = make_synthetic_explanation('positive', 'short', 'Great product loved it')
print('Demo explanation:', _demo_expl)

### Section 30: Decoder Dataset & DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader as DL

class DecoderDataset(Dataset):
    """
    Each sample is a full decoder sequence:
      [SOS] prompt ... explanation [EOS] [PAD...]
    Input  = sequence[:-1]
    Target = sequence[1:]   (next-token prediction)
    """

    def __init__(self, df, retriever, vocab, max_len, k=2):
        self.samples = []
        sent_map  = {0: 'negative', 1: 'neutral', 2: 'positive'}
        feat_map  = {0: 'short', 1: 'medium', 2: 'long'}

        print(f"Building decoder dataset ({len(df)} samples)...")
        # We can't call the full retriever per row during __init__ for large sets,
        # so we build a simple lookup: use training embeddings if available.
        for idx in range(len(df)):
            row         = df.iloc[idx]
            review_text = row['review_text']
            sent_lbl    = sent_map[row['sentiment']]
            feat_lbl    = feat_map[row['length_label']]
            expl        = make_synthetic_explanation(sent_lbl, feat_lbl, review_text)

            # Retrieve k similar context reviews (excluding itself)
            if retriever is not None:
                q_vec  = embed_text(review_text, model, vocab, DEVICE)
                top_k  = retriever.retrieve(q_vec, k=k + 1)   # +1 to skip self
                ctx    = [r['text'][:60] for r in top_k if r['text'] != review_text][:k]
            else:
                ctx = []

            seq = build_prompt(review_text, sent_lbl, feat_lbl, ctx, expl, vocab, max_len)
            self.samples.append(seq)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        seq   = torch.tensor(self.samples[i], dtype=torch.long)
        inp   = seq[:-1]
        target= seq[1:]
        return inp, target


print("DecoderDataset class defined.")

In [ ]:
import math as _math

# Build datasets — use retriever for train/val; None for speed on large sets
print("Building train decoder dataset...")
dec_train_ds = DecoderDataset(train_df.sample(min(8000, len(train_df)), random_state=42).reset_index(drop=True),
                               retriever, vocab, DEC_MAX_LEN, k=DEC_K_CONTEXT)
print("Building val decoder dataset...")
dec_val_ds   = DecoderDataset(val_df.sample(min(2000, len(val_df)), random_state=42).reset_index(drop=True),
                               retriever, vocab, DEC_MAX_LEN, k=DEC_K_CONTEXT)

dec_train_loader = DL(dec_train_ds, batch_size=DEC_BATCH, shuffle=True,  drop_last=True)
dec_val_loader   = DL(dec_val_ds,   batch_size=DEC_BATCH, shuffle=False, drop_last=False)

print(f"Train batches : {len(dec_train_loader)}")
print(f"Val   batches : {len(dec_val_loader)}")

### Section 31: Training Pipeline

In [ ]:
dec_model = DecoderTransformer(
    vocab_size  = len(vocab),
    d_model     = DEC_D_MODEL,
    num_heads   = DEC_NUM_HEADS,
    d_ff        = DEC_D_FF,
    num_layers  = DEC_NUM_LAYERS,
    max_len     = DEC_MAX_LEN,
    dropout     = DEC_DROPOUT,
).to(DEVICE)

dec_optimizer = torch.optim.AdamW(dec_model.parameters(), lr=DEC_LR, weight_decay=1e-2)
dec_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(dec_optimizer, T_max=DEC_EPOCHS)
dec_criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx)

n_params = sum(p.numel() for p in dec_model.parameters() if p.requires_grad)
print(f"Decoder parameters: {n_params:,}")

In [ ]:
def train_decoder_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for inp, target in tqdm(loader, desc="  [DEC train]", leave=False):
        inp, target = inp.to(device), target.to(device)
        pad_mask = (inp == vocab.pad_idx)
        logits = model(inp, pad_mask=pad_mask)          # (B, T-1, V)
        loss   = criterion(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def eval_decoder(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for inp, target in tqdm(loader, desc="  [DEC val] ", leave=False):
            inp, target = inp.to(device), target.to(device)
            pad_mask = (inp == vocab.pad_idx)
            logits = model(inp, pad_mask=pad_mask)
            loss   = criterion(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(loader)


dec_history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

print("Training decoder...")
for epoch in range(1, DEC_EPOCHS + 1):
    tr_loss  = train_decoder_epoch(dec_model, dec_train_loader, dec_optimizer, dec_criterion, DEVICE)
    val_loss = eval_decoder(dec_model, dec_val_loader, dec_criterion, DEVICE)
    dec_scheduler.step()
    dec_history['train_loss'].append(tr_loss)
    dec_history['val_loss'].append(val_loss)
    tr_ppl  = _math.exp(tr_loss)
    val_ppl = _math.exp(val_loss)
    print(f"Epoch {epoch}/{DEC_EPOCHS}  "
          f"train_loss={tr_loss:.4f}  train_ppl={tr_ppl:.2f}  "
          f"val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(dec_model.state_dict(), MODELS_DIR / 'decoder_model.pt')
        print(f"  => best model saved (val_loss={val_loss:.4f})")

print("Training complete.")

### Section 32: Decoder Learning Curves

In [ ]:
ep_x = range(1, len(dec_history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(ep_x, dec_history['train_loss'], 'o-', color='steelblue',  label='Train')
axes[0].plot(ep_x, dec_history['val_loss'],   's--',color='darkorange', label='Val')
axes[0].set_title('Decoder Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep_x, [_math.exp(l) for l in dec_history['train_loss']], 'o-',  color='steelblue',  label='Train')
axes[1].plot(ep_x, [_math.exp(l) for l in dec_history['val_loss']],   's--', color='darkorange', label='Val')
axes[1].set_title('Decoder Perplexity', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'decoder_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Decoder learning curves saved.")

### Section 33: Perplexity on Test Set

In [ ]:
# Build test decoder dataset
print("Building test decoder dataset...")
dec_test_ds = DecoderDataset(
    test_df.sample(min(1000, len(test_df)), random_state=42).reset_index(drop=True),
    retriever, vocab, DEC_MAX_LEN, k=DEC_K_CONTEXT
)
dec_test_loader = DL(dec_test_ds, batch_size=DEC_BATCH, shuffle=False, drop_last=False)

# Load best checkpoint
dec_model.load_state_dict(torch.load(MODELS_DIR / 'decoder_model.pt', map_location=DEVICE))
test_loss = eval_decoder(dec_model, dec_test_loader, dec_criterion, DEVICE)
test_ppl  = _math.exp(test_loss)

print(f"Test Cross-Entropy Loss : {test_loss:.4f}")
print(f"Test Perplexity         : {test_ppl:.2f}")

# Save metrics
dec_metrics = {'test_loss': test_loss, 'test_perplexity': test_ppl}
with open(RESULTS_DIR / 'decoder_metrics.json', 'w') as f:
    _json.dump(dec_metrics, f, indent=2)
print("Decoder metrics saved.")

### Section 34: Autoregressive Generation

In [ ]:
@torch.no_grad()
def generate_explanation(review_text: str,
                         sent_label: str,
                         feat_label: str,
                         context_texts: list,
                         vocab,
                         model,
                         device,
                         max_new_tokens: int = 40,
                         temperature: float = 0.8) -> str:
    """
    Greedily generate tokens after the prompt, stopping at EOS or max_new_tokens.
    """
    model.eval()
    # Build prompt (without explanation part)
    ctx_part = ' '.join(context_texts) if context_texts else 'none'
    prompt   = (f"review {review_text} sentiment {sent_label} "
                f"feature {feat_label} context {ctx_part} explanation")
    ids   = vocab.numericalize(prompt, add_special=False)
    ids   = [vocab.sos_idx] + ids
    # Truncate prompt if too long (leave room for new tokens)
    max_prompt = DEC_MAX_LEN - max_new_tokens - 1
    ids = ids[:max_prompt]

    seq = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)  # (1, T)

    generated = []
    for _ in range(max_new_tokens):
        if seq.size(1) >= DEC_MAX_LEN:
            break
        pad_mask = (seq == vocab.pad_idx)
        logits   = model(seq, pad_mask=pad_mask)   # (1, T, V)
        next_logits = logits[0, -1, :] / temperature
        next_token  = torch.argmax(next_logits).item()
        if next_token == vocab.eos_idx:
            break
        generated.append(next_token)
        seq = torch.cat([seq, torch.tensor([[next_token]], device=device)], dim=1)

    return vocab.decode(generated)


print("generate_explanation() defined.")

### Section 35: Qualitative Examples — 5 Generated Explanations

In [ ]:
SENT_MAP_INV = {0: 'negative', 1: 'neutral', 2: 'positive'}
FEAT_MAP_INV = {0: 'short', 1: 'medium', 2: 'long'}

print("=" * 72)
print("QUALITATIVE EXAMPLES — DECODER GENERATED EXPLANATIONS")
print("=" * 72)

sample_indices = test_df.sample(5, random_state=7).index.tolist()

for rank, idx in enumerate(sample_indices, 1):
    row         = test_df.loc[idx]
    review_text = row['review_text']
    sent_lbl    = SENT_MAP_INV[row['sentiment']]
    feat_lbl    = FEAT_MAP_INV[row['length_label']]

    # Retrieve context
    q_vec    = embed_text(review_text, model, vocab, DEVICE)
    top_k    = retriever.retrieve(q_vec, k=DEC_K_CONTEXT + 1)
    ctx_txts = [r['text'][:60] for r in top_k if r['text'] != review_text][:DEC_K_CONTEXT]

    explanation = generate_explanation(
        review_text, sent_lbl, feat_lbl, ctx_txts, vocab, dec_model, DEVICE
    )

    print(f"\n[Example {rank}]")
    print(f"  Review   : {review_text[:120]}...")
    print(f"  Sentiment: {sent_lbl}  |  Length feature: {feat_lbl}")
    print(f"  Context 1: {ctx_txts[0] if ctx_txts else 'N/A'}")
    print(f"  Generated: {explanation}")
print("=" * 72)

### Section 36: RAG Ablation Study

We compare the model's **test perplexity** when the decoder is given retrieved context (full RAG system) vs. when context is removed (no-retrieval baseline).

In [ ]:
class DecoderDatasetNoContext(Dataset):
    """Same as DecoderDataset but context is always empty (ablation baseline)."""

    def __init__(self, df, vocab, max_len):
        self.samples = []
        sent_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
        feat_map = {0: 'short',    1: 'medium',  2: 'long'}
        for idx in range(len(df)):
            row      = df.iloc[idx]
            sent_lbl = sent_map[row['sentiment']]
            feat_lbl = feat_map[row['length_label']]
            expl     = make_synthetic_explanation(sent_lbl, feat_lbl, row['review_text'])
            seq      = build_prompt(row['review_text'], sent_lbl, feat_lbl,
                                    [], expl, vocab, max_len)   # empty context
            self.samples.append(seq)

    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        seq = torch.tensor(self.samples[i], dtype=torch.long)
        return seq[:-1], seq[1:]


abl_df = test_df.sample(min(1000, len(test_df)), random_state=42).reset_index(drop=True)

# Baseline: no context
abl_no_ctx_ds     = DecoderDatasetNoContext(abl_df, vocab, DEC_MAX_LEN)
abl_no_ctx_loader = DL(abl_no_ctx_ds, batch_size=DEC_BATCH, shuffle=False)
no_ctx_loss = eval_decoder(dec_model, abl_no_ctx_loader, dec_criterion, DEVICE)
no_ctx_ppl  = _math.exp(no_ctx_loss)

# Full RAG: with context (already computed above as test_ppl)
print("\n" + "=" * 55)
print("RAG ABLATION STUDY")
print("=" * 55)
print(f"{'System':<30} {'Loss':>8} {'Perplexity':>12}")
print("-" * 55)
print(f"{'Baseline (no retrieval)':<30} {no_ctx_loss:>8.4f} {no_ctx_ppl:>12.2f}")
print(f"{'Full RAG (with context)':<30} {test_loss:>8.4f} {test_ppl:>12.2f}")
print("-" * 55)
improvement = no_ctx_ppl - test_ppl
print(f"Perplexity reduction from RAG: {improvement:.2f}")
print("=" * 55)

# Save ablation results
ablation_results = {
    'no_context_loss': no_ctx_loss, 'no_context_perplexity': no_ctx_ppl,
    'rag_loss': test_loss,          'rag_perplexity': test_ppl,
    'perplexity_improvement': improvement
}
with open(RESULTS_DIR / 'ablation_results.json', 'w') as f:
    _json.dump(ablation_results, f, indent=2)
print("Ablation results saved.")